# <center> Elementy numerycznej algebry liniowej </center>

Rozwiązywanie układów równań liniowych jest jednym z podstawowych problemów metod numerycznych. Układy równań liniowych występują w wielu dziedzinach nauki i inżynierii. Stosuje się też w uczeniu maszynowym np. podczas regresji z błędem średniokwadratowym. 


Istnieje kilka metod rozwiązywania układów równań. Na dzisiejszych zajęciach zajmiemy się:
* eliminacją Gaussa bez oraz z wyborem elementu głównego,
* metodami iteracyjnymi.

Problem rozwiązywania układu równań liniowych będzie nam towarzyszły do końca zajęć z tego przedmiotu.

## Normy i wskaźniki uwarunkowania

Wrażliwość układu (zmiana rozwiązania) na niewielkie zaburzenia wektora `b` zależy od macierzy `A` i ocenia się ja za pomocą tzw. współczynnika lub [wskaźnika uwarunkowania macierzy](https://pl.wikipedia.org/wiki/Wskaźnik_uwarunkowania) (ang. *condition number*). Im wyższa wartość tego wskaźnika. tym macierz jest gorzej uwarunkowana. Wskaźnik uwarunkowania to iloczyn normy macierzy z normą jej odwrotności.

$$cond(A)=|A|_{p}\cdot|A^{-1}|_{p}$$
gdzie *p* oznacza jedną z norm macierzy.

In [3]:
import numpy as np
import scipy
import matplotlib.pyplot as plt

***Zadanie 1.***

Porównaj normy 1,2, $\infty$ następujących macierzy:
* [Hilberta](https://pl.wikipedia.org/wiki/Macierz_Hilberta): o wymiarach 5x5 i 15x15
* [Vandermonde'a](https://pl.wikipedia.org/wiki/Macierz_Vandermonde’a): o wymiarach 5x5 i 15x15
* losowej o wartościach z przedziału [0,1]:  o wymiarach 5x5 i 15x15
* $P=\left[\begin{array}{cccc}4 & 1 & -1 & 0 \\ 1 & 3 & -1 & 0 \\ -1 & -1 & 5 & 2 \\ 0 & 0 & 2 & 4\end{array}\right]$

Czy wśród powyższych macierzy jest macierz [diagonalnie dominująca](https://pl.wikipedia.org/wiki/Macierz_przekątniowo_dominująca)?


In [4]:
sizes = [5, 15]
matrices = {}

for n in sizes:
    matrices[f'Hilbert {n}x{n}'] = scipy.linalg.hilbert(n)

for n in sizes:
    v = np.linspace(1, 2, n)
    matrices[f'Vandermonde {n}x{n}'] = np.vander(v, increasing=True)

for n in sizes:
    matrices[f'Losowa {n}x{n}'] = np.random.rand(n, n)

P = np.array([[4, 1, -1, 0],
              [1, 3, -1, 0],
              [-1, -1, 5, 2],
              [0, 0, 2, 4]], dtype=float)
matrices['Macierz P'] = P

def is_diagonally_dominant(A):
    D = np.abs(np.diag(A))
    S = np.sum(np.abs(A), axis=1) - D
    return np.all(D >= S)

for name, A in matrices.items():
    norm1 = np.linalg.norm(A, 1)
    norm2 = np.linalg.norm(A, 2)
    norm_inf = np.linalg.norm(A, np.inf)
    diag_dom = is_diagonally_dominant(A)
    
    print(f"--- {name} ---")
    print(f"Norma 1: {norm1:.2e} | Norma 2: {norm2:.2e} | Norma inf: {norm_inf:.2e}")
    print(f"Diagonalnie dominująca: {'TAK' if diag_dom else 'NIE'}\n")

--- Hilbert 5x5 ---
Norma 1: 2.28e+00 | Norma 2: 1.57e+00 | Norma inf: 2.28e+00
Diagonalnie dominująca: NIE

--- Hilbert 15x15 ---
Norma 1: 3.32e+00 | Norma 2: 1.85e+00 | Norma inf: 3.32e+00
Diagonalnie dominująca: NIE

--- Vandermonde 5x5 ---
Norma 1: 3.39e+01 | Norma 2: 2.30e+01 | Norma inf: 3.10e+01
Diagonalnie dominująca: NIE

--- Vandermonde 15x15 ---
Norma 1: 3.95e+04 | Norma 2: 2.37e+04 | Norma inf: 3.28e+04
Diagonalnie dominująca: NIE

--- Losowa 5x5 ---
Norma 1: 3.52e+00 | Norma 2: 2.76e+00 | Norma inf: 3.61e+00
Diagonalnie dominująca: NIE

--- Losowa 15x15 ---
Norma 1: 8.56e+00 | Norma 2: 7.60e+00 | Norma inf: 9.84e+00
Diagonalnie dominująca: NIE

--- Macierz P ---
Norma 1: 9.00e+00 | Norma 2: 7.09e+00 | Norma inf: 9.00e+00
Diagonalnie dominująca: TAK



*Wskazówka: Do wyznaczenia norm możesz wykorzystać funkcję `numpy.linalg.norm`*

***Zadanie 2.***

Oblicz wskaźniki uwarunkowania macierzy z poprzedniego zadania.

*Wskazówka: Możesz wykorzystać funkcję `numpy.linalg.cond`.*

In [5]:
for name, A in matrices.items():
    cond = np.linalg.cond(A) 
    
    print(f"--- {name} ---")
    print(f"Wskaźnik uwarunkowania cond(A): {cond:.2e}\n")

--- Hilbert 5x5 ---
Wskaźnik uwarunkowania cond(A): 4.77e+05

--- Hilbert 15x15 ---
Wskaźnik uwarunkowania cond(A): 7.36e+17

--- Vandermonde 5x5 ---
Wskaźnik uwarunkowania cond(A): 4.08e+04

--- Vandermonde 15x15 ---
Wskaźnik uwarunkowania cond(A): 3.08e+18

--- Losowa 5x5 ---
Wskaźnik uwarunkowania cond(A): 5.19e+01

--- Losowa 15x15 ---
Wskaźnik uwarunkowania cond(A): 1.20e+02

--- Macierz P ---
Wskaźnik uwarunkowania cond(A): 3.54e+00



## Rozwiązywanie układów równań metodą eliminacji Gaussa

***Zadanie 3.***

Jedną z metod rozwiązywania układów równań liniowych jest metoda eliminacji Gaussa. Metoda ta występuje w kilku odmianach. Poza podstawowym wariantem, możliwe jest zastosowanie metody z wyborem elementu głownego (tzw. *pivoting*). 

Celem tego zadania jest porównanie błędów rozwiązania otrzymanego z tych dwóch wariantów eliminacji Gaussa. Poniżej znajdują się implementacje obu tych metod. Każda z funkcji przyjmuje macierz `A` oraz wektor prawej strony równania `b`.

Samo polecenie znajduje się poniżej.

In [8]:
def gauss_pivot(A, b):
    A=A.copy()
    b=b.copy()
    n = len(b)
    for k in range(n-1):
        ind_max = k
        for j in range(k+1, n):
            if abs(A[j,k]) > abs(A[ind_max,k]):
                ind_max = j
        if ind_max > k:
            tmp = A[ind_max,k:n].copy()
            A[ind_max,k:n] = A[k,k:n]
            A[k,k:n] = tmp
            tmp = b[ind_max].copy()
            b[ind_max] = b[k]
            b[k] = tmp
        akk = A[k,k]
        l = A[k+1:n,k] / akk
        for i in range(k+1, n):
            A[i,k] = 0
            A[i,k+1:n] = A[i,k+1:n] - l[i-k-1] * A[k,k+1:n]
            b[i] = b[i] - l[i-k-1] * b[k]
    x = np.zeros(n)
    x[n-1] = b[n-1]/A[n-1,n-1]
    for k in range(n-2, -1, -1):
        x[k] = (b[k] - np.dot(A[k,k+1:n], x[k+1:n])) / A[k,k]
    return x

In [7]:
def gauss(A, b):
    A=A.copy()
    b=b.copy()
    n = len(b)
    for k in range(n-1):
        akk = A[k,k]
        l = A[k+1:n,k] / akk
        for i in range(k+1, n):
            A[i,k] = 0
            A[i,k+1:n] = A[i,k+1:n] - l[i-k-1] * A[k,k+1:n]
            b[i] = b[i] - l[i-k-1] * b[k]
    x = np.zeros(n)
    x[n-1] = b[n-1] / A[n-1,n-1]
    for k in range(n-2, -1, -1):
        x[k] = (b[k] - np.dot(A[k,k+1:n], x[k+1:n])) / A[k,k]
    return x

Stwórz macierze wartości losowych `A` o wymiarach 10x10 oraz wektor `b` o odpowiednich wymiarach. 
Chcemy rozwiązać układ równań `Ax=b` metodami eliminacji Gaussa bez oraz z wyborem elementu głównego, a następnie porównać dokładność wyników. Metoda z wyborem elementu głównego powinna dawać mniejszy błąd w przypadku dużych wartości znajdujących się na przekątnej. Sprawdź czy to prawda powtarzając obliczenia z  macierzami `A` zawierającym na pierwszym elemencie przekątnej coraz to mniejsze wartości (tak aby wzrosło znaczenie dalszych elementów na przękątnej i tym samym uaktywnił się wybór innego niż pierwszy elementu głównego).

Wskazówka:Do porównania możesz wykorzystać residuum. Jeżeli `x` jest rozwiązaniem układu to `Ax` powinno być równe `b`. Residuum to różnica pomiędzy `b` oraz `Ax`: `res=|b-Ax|`. Możesz porównać zawartości poszczególnych elementów lub obliczyć jakąś normę z otrzymanego wektora.

In [9]:
n = 10
A = np.random.rand(n, n)
x_true = np.random.rand(n)
b = A @ x_true

print("Standardowa macierz losowa")
x_gauss = gauss(A, b)
x_pivot = gauss_pivot(A, b)
print(f"Residuum - bez wyboru (Gauss):  {np.linalg.norm(b - A @ x_gauss):.4e}")
print(f"Residuum - z wyborem (Pivot):   {np.linalg.norm(b - A @ x_pivot):.4e}")

A[0, 0] = 1e-16
b = A @ x_true

print("\nMacierz z bardzo małym elementem A[0,0]")
x_gauss_bad = gauss(A, b)
x_pivot_bad = gauss_pivot(A, b)
print(f"Residuum - bez wyboru (Gauss):  {np.linalg.norm(b - A @ x_gauss_bad):.4e}")
print(f"Residuum - z wyborem (Pivot):   {np.linalg.norm(b - A @ x_pivot_bad):.4e}")

Standardowa macierz losowa
Residuum - bez wyboru (Gauss):  6.2804e-16
Residuum - z wyborem (Pivot):   5.4390e-16

Macierz z bardzo małym elementem A[0,0]
Residuum - bez wyboru (Gauss):  1.8914e+00
Residuum - z wyborem (Pivot):   4.9651e-16


## Metody iteracyjne

Innym sposobem na rozwiązanie układu równań liniowych jest wykorzystanie metod iteracyjnych, które generują ciągi przybliżeń wektora stanowiącego rozwiązanie układu. Państwa zadaniem będzie implementacja i porównanie zbieżności trzech najpopularniejszych metod iteracyjnego rozwiązywania układów równań liniowych

***Zadanie 4.***

Porównanie zbieżności metod Jacobiego, Gaussa-Seidla i Younga (SOR).
* Zaimplementuj solvery rozwiązujące układy równań metodami Jacobiego, Gaussa-Seidela  i Younga (SOR). Każda funkcja powinna przyjmować macierz A i wektor prawej strony b. Dla uproszczenia, dopuszczalne jest wykorzystanie  inv dla obliczenia macierzy odwrotnej do macierzy trójkątnej (w metodzie G-S i Younga).
* Porównaj zbieżność ciągów iteracyjnych otrzymanych 3 metodami dla 3 układów równań (3 macierzy). W metodzie Younga możesz przyjąć np. $ω = 1.2$.
* Dla macierzy, dla której metoda Younga okazała się zbieżna, porównaj zbieżność ciągów iteracyjnych otrzymanych dla wartości $0 < ω < 3$ (dodatkowe).
* Dla jakiej wartości parametru $ω$ zbieżność ciągu iteracyjnego jest najlepsza? Wynik otrzymany na podstawie obserwacji ciągu odchyleń od rozwiązania dokładnego należy porównać z wnioskiem płynącym z wykresu zależności promienia spektralnego macierzy iteracji w zależności od parametru $ω$ (dodatkowe).

In [10]:
def jacobi(A, b, max_iter=100, tol=1e-10):
  D = np.diag(np.diag(A))
  R = A - D
  x = np.zeros_like(b)
  for i in range(max_iter):
      x_new = np.linalg.inv(D) @ (b - R @ x)
      if np.linalg.norm(x_new - x) < tol: return x_new, i+1
      x = x_new
  return x, max_iter

def gauss_seidel(A, b, max_iter=100, tol=1e-10):
  L = np.tril(A)
  U = A - L
  x = np.zeros_like(b)
  for i in range(max_iter):
      x_new = np.linalg.inv(L) @ (b - U @ x)
      if np.linalg.norm(x_new - x) < tol: return x_new, i+1
      x = x_new
  return x, max_iter

def sor(A, b, omega=1.2, max_iter=100, tol=1e-10):
  D = np.diag(np.diag(A))
  L = np.tril(A, -1)
  U = np.triu(A, 1)
  M = D + omega * L
  N = (1 - omega) * D - omega * U
  x = np.zeros_like(b)
  for i in range(max_iter):
      x_new = np.linalg.inv(M) @ (omega * b + N @ x)
      if np.linalg.norm(x_new - x) < tol: return x_new, i+1
      x = x_new
  return x, max_iter

P_b = np.array([1, 2, 3, 4], dtype=float)

_, iter_j = jacobi(P, P_b)
_, iter_gs = gauss_seidel(P, P_b)
_, iter_sor = sor(P, P_b, omega=1.2)

print("Liczba iteracji do zbieżności dla Macierzy P:")
print(f"Jacobi: {iter_j}")
print(f"Gauss-Seidel: {iter_gs}")
print(f"SOR (w=1.2): {iter_sor}")

Liczba iteracji do zbieżności dla Macierzy P:
Jacobi: 52
Gauss-Seidel: 19
SOR (w=1.2): 18


## Porównanie rozwiązania za pomocą metody `solve` oraz z użyciem odwrotności na przykładzie macierzy źle uwarunkowanej

***Zadanie 5.***

Dany jest układ równań $Hx=b$.
* H jest macierzą Hilberta o wymiarach $n=5x5$ (I przypadek) i $n=15x15$ (II przypadek),
* b jest wektorem o następujących elementach $b_i = 1/(n + i + 1)$ Uwaga: $i=1,\dots,n$.

Do rozwiązania układu wykorzystaj dwa algorytmy:
1. Z odwracaniem macierzy współczynników H.
2. Metodę `numpy.linalg.solve`.

Porównaj błędy obu rozwiązań. Aby ocenić błąd możesz:
* wyznaczyć wektor residuum otrzymanego rozwiązania,
* rozwiązać układ równań z innym wektorem $b$. Załóż, że wektor rozwiązania ma wszystkie elementy (współrzędne) równe 1 ($u_i = 1, i = 1, 2, . . . , n$). Wtedy $b = Hu$. Układ rozwiążemy bez korzystania z wiedzy o postaci $u$. Dopiero wynik porównamy ze znanym nam $u$.

In [11]:
for n in [5, 15]:
  H = scipy.linalg.hilbert(n)
  u = np.ones(n)
  b = H @ u
  
  x_solve = np.linalg.solve(H, b)
  err_solve = np.linalg.norm(x_solve - u)

  x_inv = np.linalg.inv(H) @ b
  err_inv = np.linalg.norm(x_inv - u)
  
  print(f"\n--- Macierz Hilberta {n}x{n} ---")
  print(f"Błąd metody solve():       {err_solve:.4e}")
  print(f"Błąd metody z np.inv():    {err_inv:.4e}")


--- Macierz Hilberta 5x5 ---
Błąd metody solve():       8.4589e-12
Błąd metody z np.inv():    2.9418e-11

--- Macierz Hilberta 15x15 ---
Błąd metody solve():       9.7953e+00
Błąd metody z np.inv():    1.8140e+03


**Zadanie domowe. Znaczenie wskaźnika uwarunkowania macierzy w szacowaniu błędu rozwiązania**


Dana jest następująca macierz A współczynników układu dwóch równań liniowy:
$$A=\begin{bmatrix}10^5 & 9.9\cdot10^4\\1.00001& 0.99\end{bmatrix}$$

Wektor prawej strony równania $Ax=b$ dla rozwiązania x = $[1, 1]^T$ możemy wyznaczyć z równości $b = Ax$.

Należy:
* obliczyć wskaźnik uwarunkowania macierzy $A$,
* rozwiązać układ równań $Ax = b$ (nie korzystając z wiedzy o przyjętym rozwiązaniu dokładnym x) korzystając z funkcji `np.linalg.solve`,
* ocenić błąd otrzymanego rozwiązania i porównać go z błędem szacowanym za pomocą wskaźnika uwarunkowania macierzy A,
* przeprowadzić skalowanie tak, aby macierz $A$ była wyważona wierszami,
* wyznaczyć nowe wartości wektora b tak, aby rozwiązanie dokładne się nie
zmieniło,
* obliczyć wskaźnik uwarunkowania macierzy skalowanej,
* rozwiązać układ równań tą samą metodą jak poprzednio,
* ocenić błąd otrzymanego rozwiązania i porównać go z błędem szacowanym za pomocą wskaźnika uwarunkowania skalowanej macierzy $A$.
1. Czy błąd numeryczny rozwiązania w obu przypadkach jest tego samego rzędu?
2. Które szacowanie błędu jest bardziej zbliżone do faktycznego błędu?